In [ ]:
!pip install -qU langchain-community pypdf
!pip install -qU langchain langchain-huggingface sentence_transformers


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from pprint import pprint

In [ ]:
file_path="/Attention Is All You Need - 2017 (1706.03762).pdf"
loader=PyPDFLoader(file_path)
doc=loader.load()
pprint(doc[0].metadata)
pprint(doc[0].page_content)



In [ ]:
!pip install -qU langchain-text-splitters


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_config=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
all_splits=text_config.split_documents(doc)
pprint(len(all_splits))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
!pip install -qU langchain-chroma

In [ ]:
from langchain_chroma import Chroma
vector_store=Chroma(
    collection_name="attention_google_doc",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"

)
document_ids=vector_store.add_documents(documents=all_splits)


In [ ]:
def retrieve_context(query: str,k: int=2):
  retrieved_docs=vector_store.similarity_search(query,k=k)

  docs_content=""
  for i in retrieved_docs:
    docs_content+=f"source:{i.metadata}\n"
    docs_content+=f"Content:{i.page_content}\n\n"

  return  docs_content,retrieved_docs

In [ ]:
!pip install -U  langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
api_key=userdata.get("GEMINI_API_KEY")
model=init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key,
)


In [ ]:
def docu_chat(user_query):
  context,source_docs=retrieve_context(user_query,k=2)
  system_message=f"""You are a helpful chatbot.
                     Use only the following pieces of context to answer the
                     question. Don't makeup any new information: {context} """
  message=[
      {"role":"system","content":system_message},
      {"role":"user","content":user_query},
  ]
  response=model.invoke(message)
  return {
      "answer":response.content,
      "source_data":source_docs,
      "content_used":context
  }

In [ ]:
print(docu_chat("explain what is the use of decoders in transformer")["answer"])